In [40]:
import pandas as pd
import numpy as np

import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

Load food dataset

In [2]:
food_df = pd.read_csv("../data/processed/food_nutrition.csv")

In [3]:
food_df.head()

food_df.info()

food_df.columns

<class 'pandas.DataFrame'>
RangeIndex: 2395 entries, 0 to 2394
Data columns (total 35 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   food                  2395 non-null   str    
 1   caloric_value         2395 non-null   int64  
 2   fat                   2395 non-null   float64
 3   saturated_fats        2395 non-null   float64
 4   monounsaturated_fats  2395 non-null   float64
 5   polyunsaturated_fats  2395 non-null   float64
 6   carbohydrates         2395 non-null   float64
 7   sugars                2395 non-null   float64
 8   protein               2395 non-null   float64
 9   dietary_fiber         2395 non-null   float64
 10  cholesterol           2395 non-null   float64
 11  sodium                2395 non-null   float64
 12  water                 2395 non-null   float64
 13  vitamin_a             2395 non-null   float64
 14  vitamin_b1            2395 non-null   float64
 15  vitamin_b11           2395 non-n

Index(['food', 'caloric_value', 'fat', 'saturated_fats',
       'monounsaturated_fats', 'polyunsaturated_fats', 'carbohydrates',
       'sugars', 'protein', 'dietary_fiber', 'cholesterol', 'sodium', 'water',
       'vitamin_a', 'vitamin_b1', 'vitamin_b11', 'vitamin_b12', 'vitamin_b2',
       'vitamin_b3', 'vitamin_b5', 'vitamin_b6', 'vitamin_c', 'vitamin_d',
       'vitamin_e', 'vitamin_k', 'calcium', 'copper', 'iron', 'magnesium',
       'manganese', 'phosphorus', 'potassium', 'selenium', 'zinc',
       'nutrition_density'],
      dtype='str')

The ML model predicts nutritional requirements, and a ranking engine computes a weighted similarity score between the predicted nutrition vector and every food item. Foods are then ranked by similarity, with dietary preferences and allergies applied as filters before selecting the top recommendations.

Cosine similarity measures how close these vectors are in multidimensional space.

Benefits:

Used in recommendation systems.

Used in semantic search.

Used in retrieval systems.

More scalable than manually tuning weights.

In [4]:
recommendation_df = food_df[
    [
        "food",
        "caloric_value",
        "protein",
        "carbohydrates",
        "fat",
        "dietary_fiber",
        "sugars",
        "sodium"
    ]
].copy()

Why .copy()?

Without .copy(),

Pandas may give

SettingWithCopyWarning

Later.

Creating a copy is considered good practice because we'll modify this DataFrame.

In [5]:
recommendation_df.head()

,food,caloric_value,protein,carbohydrates,fat,dietary_fiber,sugars,sodium
0,cream cheese,51,0.9,0.8,5.0,0.0,0.500,0.016
1,neufchatel cheese,215,7.8,3.1,19.4,0.0,2.700,0.300
2,requeijao cremoso light catupiry,49,0.8,0.9,3.6,0.1,3.400,0.000
3,ricotta cheese,30,1.5,1.5,2.0,0.0,0.091,0.017
4,cream cheese low fat,30,1.2,1.2,2.3,0.0,0.900,0.046


In [6]:
nutrition_features = [
    "caloric_value",
    "protein",
    "carbohydrates",
    "fat",
    "dietary_fiber",
    "sugars",
    "sodium"
]

Scale Nutrition Values

This is where vector search begins.

In [8]:
scaler = StandardScaler()

food_vectors = scaler.fit_transform(
    recommendation_df[nutrition_features]
)

In [9]:
food_vectors.shape

(2395, 7)

Create a Sample Patient

In [10]:
sample_patient = pd.DataFrame(
    {
        "age": [45],
        "gender": ["Male"],
        "height_m": [1.72],
        "weight_kg": [85],
        "bmi": [28.7],
        "health_condition": ["Diabetes"],
        "activity_level": ["Medium"],
        "diet_type": ["Vegetarian"],
        "allergy": ["No Allergy"]
    }
)


Why DataFrame?

Our pipeline was trained on a DataFrame with column names.

If you pass

[45, "Male", 1.72, ...]

the pipeline won't know which value belongs to which feature.

A DataFrame preserves the feature names.

Predict Nutrition

In [ ]:
predicted_targets = pipeline.predict(sample_patient)

Load the Model

In [12]:
pipeline = joblib.load(
    "../models/nutrition_model.pkl"
)

In [13]:
type(pipeline)

sklearn.pipeline.Pipeline

In [14]:
scaler = StandardScaler()
food_vectors = scaler.fit_transform(
    recommendation_df[nutrition_features]
)

In [15]:
joblib.dump(
    scaler,
    "../models/food_scaler.pkl"
)

['../models/food_scaler.pkl']

In [16]:
predicted_targets = pipeline.predict(sample_patient)

In [18]:
predicted_df = pd.DataFrame(
    predicted_targets,
    columns=[
        "caloric_value",
        "protein",
        "carbohydrates",
        "fat",
        "dietary_fiber",
        "sugars",
        "sodium"
    ]
)
predicted_df

,caloric_value,protein,carbohydrates,fat,dietary_fiber,sugars,sodium
0,1700.0,90.0,170.0,55.0,35.0,25.0,1800.0


Scale the patient vector

In [19]:
patient_vector = scaler.transform(predicted_df)

We use

transform()

NOT

fit_transform()

Reason:

The scaler has already learned the mean and standard deviation from the food dataset.

If we call fit_transform() on a single patient, it will compute a new mean and standard deviation, making the patient's vector incompatible with the food vectors.

Compute similarity

Now comes the AI part

In [21]:
similarity_scores = cosine_similarity(
    patient_vector,
    food_vectors
)

In [22]:
print(patient_vector.shape)
print(food_vectors.shape)

(1, 7)
(2395, 7)


cosine_similarity() compares this nutrition vector against all 2395 food vectors and returns a similarity score for each food.

In [23]:
similarity_scores.shape

(1, 2395)

Add Similarity Scores to the DataFrame

In [24]:
recommendation_df["similarity_score"] = similarity_scores[0]

In [25]:
recommendation_df.head()

,food,caloric_value,protein,carbohydrates,fat,dietary_fiber,sugars,sodium,similarity_score
0,cream cheese,51,0.9,0.8,5.0,0.0,0.500,0.016,-0.246035
1,neufchatel cheese,215,7.8,3.1,19.4,0.0,2.700,0.300,0.002727
2,requeijao cremoso light catupiry,49,0.8,0.9,3.6,0.1,3.400,0.000,-0.268175
3,ricotta cheese,30,1.5,1.5,2.0,0.0,0.091,0.017,-0.237046
4,cream cheese low fat,30,1.2,1.2,2.3,0.0,0.900,0.046,-0.216090


Sort Foods by Similarity

In [26]:
recommendation_df = recommendation_df.sort_values(
    by="similarity_score",
    ascending=False
)

Display the Top 10 Recommendations

In [27]:
top10 = recommendation_df.head(10)

top10[
    [
        "food",
        "caloric_value",
        "protein",
        "carbohydrates",
        "fat",
        "dietary_fiber",
        "similarity_score"
    ]
]

,food,caloric_value,protein,carbohydrates,fat,dietary_fiber,similarity_score
1943,adobo fresco,780,5.8,53.6,60.2,3.7,0.998497
1047,salt,0,0.0,0.0,0.0,0.0,0.981224
370,jellyfish dried,21,3.2,0.0,0.8,0.0,0.973963
189,beef noodle soup,168,9.6,17.9,6.2,1.5,0.972410
389,salt mackerel,415,25.2,0.0,34.1,0.0,0.968997
184,scotch broth,197,12.1,23.0,6.4,3.0,0.967917
199,chicken dumplings soup,235,13.6,14.7,13.4,1.2,0.967735
206,beef mushroom soup,186,14.0,15.9,7.3,0.6,0.961367
200,chicken vegetable soup,148,7.1,17.0,5.6,1.7,0.961199
238,chicken noodle soup,145,7.2,18.4,4.7,2.7,0.957603


Cosine Similarity only compares the direction of the nutrition vector, not whether a food is actually appropriate for a diabetic or whether it is a complete meal.

Because of scaling and vector direction, salt can sometimes receive an unexpectedly high similarity score.

This is exactly why real recommendation systems never rely only on cosine similarity.

Remove invalid foods(like salt, spices and jellyfish)

In [28]:
INVALID_FOODS = [
    "salt",
    "water",
    "vinegar",
    "spices",
    "seasoning"
]

In [29]:
recommendation_df = recommendation_df[
    ~recommendation_df["food"].str.lower().isin(INVALID_FOODS)
]

Remove Foods with Almost No Nutrition

A recommendation like salt or spices isn't useful as a meal.

In [30]:
recommendation_df = recommendation_df[
    recommendation_df["caloric_value"] > 30
]

Minimum Protein Filter

Meals should contain some protein.

In [31]:
recommendation_df = recommendation_df[
    recommendation_df["protein"] >= 3
]

Fiber Filter to meet minimum fiber requirements

In [32]:
recommendation_df = recommendation_df[
    recommendation_df["dietary_fiber"] >= 1
]

Re-sort

In [33]:
recommendation_df = recommendation_df.sort_values(
    by="similarity_score",
    ascending=False
)

Display Top 10 

In [34]:
recommendation_df.head(10)

,food,caloric_value,protein,carbohydrates,fat,dietary_fiber,sugars,sodium,similarity_score
1943,adobo fresco,780,5.8,53.6,60.2,3.7,5.8,49.4,0.998497
189,beef noodle soup,168,9.6,17.9,6.2,1.5,5.2,1.6,0.972410
184,scotch broth,197,12.1,23.0,6.4,3.0,0.0,2.1,0.967917
199,chicken dumplings soup,235,13.6,14.7,13.4,1.2,1.4,1.8,0.967735
200,chicken vegetable soup,148,7.1,17.0,5.6,1.7,3.0,1.7,0.961199
238,chicken noodle soup,145,7.2,18.4,4.7,2.7,0.0,2.1,0.957603
195,cream of mushroom soup,199,3.4,17.1,13.4,1.8,1.0,1.7,0.944469
227,minestrone soup,167,8.6,22.6,5.0,2.0,3.7,1.3,0.943345
240,cream of asparagus soup,174,4.6,21.5,8.2,1.0,1.8,1.7,0.941924
228,clam chowder soup,154,4.4,24.5,4.4,3.0,6.8,1.8,0.939840


The ML model predicts nutritional requirements, but cosine similarity alone may rank nutritionally incomplete or unsuitable foods. Therefore, I apply business-rule filtering to remove invalid items (such as salt or seasonings), enforce minimum nutritional quality, and then rank the remaining candidates.

Machine Learning predicts what nutrients are needed.

Business Logic ensures the recommendations are practical and safe.

Create a Nutrition Distance Score

Cosine similarity only measures vector direction.

We also want to measure how close each food is to the predicted nutrition values.

Let's compute the absolute difference between the predicted nutrition targets and each food.

In [35]:
predicted = predicted_df.iloc[0]

Calculate Differences

In [36]:
recommendation_df["calorie_diff"] = (
    recommendation_df["caloric_value"] -
    predicted["caloric_value"]
).abs()

recommendation_df["protein_diff"] = (
    recommendation_df["protein"] -
    predicted["protein"]
).abs()

recommendation_df["carb_diff"] = (
    recommendation_df["carbohydrates"] -
    predicted["carbohydrates"]
).abs()

recommendation_df["fat_diff"] = (
    recommendation_df["fat"] -
    predicted["fat"]
).abs()

recommendation_df["fiber_diff"] = (
    recommendation_df["dietary_fiber"] -
    predicted["dietary_fiber"]
).abs()

Now every food has five new columns:

Similarity

Calorie Difference

Protein Difference

Carb Difference

Fat Difference

Fiber Difference

Create a Total Nutrition Score

In [37]:
recommendation_df["nutrition_score"] = (
    recommendation_df["calorie_diff"] +
    recommendation_df["protein_diff"] +
    recommendation_df["carb_diff"] +
    recommendation_df["fat_diff"] +
    recommendation_df["fiber_diff"]
)

Smaller nutrition_score ====> Food is closer to the patient's required nutrition.

Combine Both Scores

use two signals:

1. Cosine similarity (higher is better)

2. Nutrition score (lower is better)

In [38]:
recommendation_df = recommendation_df.sort_values(
    by=[
        "similarity_score",
        "nutrition_score"
    ],
    ascending=[False, True]
)

Inspect the Results

In [39]:
recommendation_df[
    [
        "food",
        "similarity_score",
        "nutrition_score",
        "caloric_value",
        "protein",
        "carbohydrates",
        "fat",
        "dietary_fiber"
    ]
].head(10)

,food,similarity_score,nutrition_score,caloric_value,protein,carbohydrates,fat,dietary_fiber
1943,adobo fresco,0.998497,1157.1,780,5.8,53.6,60.2,3.7
189,beef noodle soup,0.972410,1846.8,168,9.6,17.9,6.2,1.5
184,scotch broth,0.967917,1808.5,197,12.1,23.0,6.4,3.0
199,chicken dumplings soup,0.967735,1772.1,235,13.6,14.7,13.4,1.2
200,chicken vegetable soup,0.961199,1870.6,148,7.1,17.0,5.6,1.7
238,chicken noodle soup,0.957603,1872.0,145,7.2,18.4,4.7,2.7
195,cream of mushroom soup,0.944469,1815.3,199,3.4,17.1,13.4,1.8
227,minestrone soup,0.943345,1844.8,167,8.6,22.6,5.0,2.0
240,cream of asparagus soup,0.941924,1840.7,174,4.6,21.5,8.2,1.0
228,clam chowder soup,0.939840,1859.7,154,4.4,24.5,4.4,3.0


What is MinMaxScaler?

It rescales values between 0 and 1.

Formula:

normalized = (value - min) / (max - min)

Create the Scaler

In [41]:
score_scaler = MinMaxScaler()

Normalize Similarity Score

In [42]:
recommendation_df["similarity_normalized"] = score_scaler.fit_transform(
    recommendation_df[["similarity_score"]]
)

[["similarity_score"]] ===> because fit_transform() expects a DataFrame, not a Series.

Normalize Nutrition Score

Small nutrition score = Good

Large nutrition score = Bad

After normalization:

0 = Best

1 = Worst

But we want larger = better, so we invert it.

In [43]:
recommendation_df["nutrition_normalized"] = 1 - score_scaler.fit_transform(
    recommendation_df[["nutrition_score"]]
)

Compute Final Score

combine both the scores

In [44]:
recommendation_df["final_score"] = (
    0.7 * recommendation_df["similarity_normalized"] +
    0.3 * recommendation_df["nutrition_normalized"]
)

Why 70% and 30%?

This is called a weighted linear combination.

We're saying:

70% importance → Similarity

30% importance → Nutrition distance

Later, we can experiment with different weights to see how they affect the recommendations.

Sort by Final Score

In [45]:
recommendation_df = recommendation_df.sort_values(
    by="final_score",
    ascending=False
)

In [46]:
recommendation_df[
    [
        "food",
        "similarity_score",
        "nutrition_score",
        "final_score",
        "caloric_value",
        "protein",
        "carbohydrates",
        "fat",
        "dietary_fiber"
    ]
].head(10)

,food,similarity_score,nutrition_score,final_score,caloric_value,protein,carbohydrates,fat,dietary_fiber
1943,adobo fresco,0.998497,1157.1,0.844867,780,5.8,53.6,60.2,3.7
199,chicken dumplings soup,0.967735,1772.1,0.725521,235,13.6,14.7,13.4,1.2
1991,salsa con queso,0.911990,1630.6,0.723432,358,7.9,27.9,23.8,1.8
184,scotch broth,0.967917,1808.5,0.719399,197,12.1,23.0,6.4,3.0
189,beef noodle soup,0.972410,1846.8,0.714981,168,9.6,17.9,6.2,1.5
195,cream of mushroom soup,0.944469,1815.3,0.707211,199,3.4,17.1,13.4,1.8
200,chicken vegetable soup,0.961199,1870.6,0.705650,148,7.1,17.0,5.6,1.7
1963,sofrito,0.820035,1483.4,0.705284,488,26.4,11.2,37.5,3.5
238,chicken noodle soup,0.957603,1872.0,0.703720,145,7.2,18.4,4.7,2.7
240,cream of asparagus soup,0.941924,1840.7,0.701683,174,4.6,21.5,8.2,1.0


Create the Rule Engine Function

In [47]:
def filter_foods(patient, predicted, food_df):

    filtered = food_df.copy()

    condition = patient["health_condition"].iloc[0]

    if condition == "Diabetes":

        filtered = filtered[
            (filtered["sugars"] <= predicted["sugars"] * 1.2) &
            (filtered["dietary_fiber"] >= predicted["dietary_fiber"] * 0.3)
        ]

    elif condition == "Hypertension":

        filtered = filtered[
            (filtered["sodium"] <= predicted["sodium"] * 1.1)
        ]

    elif condition == "Obesity":

        filtered = filtered[
            (filtered["caloric_value"] <= predicted["caloric_value"] * 0.6)
        ]

    elif condition == "Heart Disease":

        filtered = filtered[
            (filtered["cholesterol"] <= 100)
        ]

    return filtered

Apply the Rule Engine

In [48]:
candidate_foods = filter_foods(
    sample_patient,
    predicted,
    recommendation_df
)

In [49]:
print(len(candidate_foods))

95


In [50]:
candidate_foods.head()

,food,caloric_value,protein,carbohydrates,fat,dietary_fiber,sugars,sodium,similarity_score,calorie_diff,protein_diff,carb_diff,fat_diff,fiber_diff,nutrition_score,similarity_normalized,nutrition_normalized,final_score
2366,taco salad taco bell,906,35.7,80.5,48.9,16.0,0.0,1.9,0.339747,794.0,54.3,89.5,6.1,19.0,962.9,0.557372,0.593281,0.568145
1939,taco flavor tortilla chips,1090,17.9,143.2,54.9,12.0,0.0,1.8,0.248747,610.0,72.1,26.8,0.1,23.0,732.0,0.496227,0.724534,0.564719
193,black bean soup,234,12.4,39.6,3.4,17.5,6.4,2.5,0.563319,1466.0,77.6,130.4,51.6,17.5,1743.1,0.707595,0.149784,0.540252
243,bean with pork soup,335,15.3,44.1,11.5,15.3,7.8,1.7,0.440839,1365.0,74.7,125.9,43.5,19.7,1628.8,0.625298,0.214757,0.502136
188,chili with beans canned,287,14.6,30.5,14.1,11.3,3.0,1.3,0.463632,1413.0,75.4,139.5,40.9,23.7,1692.5,0.640613,0.178547,0.501993


Recompute Food Vectors for the Candidate Foods

In [51]:
nutrition_features = [
    "caloric_value",
    "protein",
    "carbohydrates",
    "fat",
    "dietary_fiber",
    "sugars",
    "sodium"
]

candidate_vectors = scaler.transform(
    candidate_foods[nutrition_features]
)

Compute Cosine Similarity

compare the patient only with these 95 foods.

In [52]:
candidate_similarity = cosine_similarity(
    patient_vector,
    candidate_vectors
)

Save the Similarity Scores

In [53]:
candidate_foods = candidate_foods.copy()

candidate_foods["similarity_score"] = candidate_similarity[0]

In [54]:
candidate_foods = candidate_foods.sort_values(
    by="similarity_score",
    ascending=False
)

In [55]:
candidate_foods[
    [
        "food",
        "caloric_value",
        "protein",
        "carbohydrates",
        "fat",
        "dietary_fiber",
        "sugars",
        "sodium",
        "similarity_score"
    ]
].head(10)

,food,caloric_value,protein,carbohydrates,fat,dietary_fiber,sugars,sodium,similarity_score
193,black bean soup,234,12.4,39.6,3.4,17.5,6.4,2.5,0.563319
188,chili with beans canned,287,14.6,30.5,14.1,11.3,3.0,1.3,0.463632
243,bean with pork soup,335,15.3,44.1,11.5,15.3,7.8,1.7,0.440839
2366,taco salad taco bell,906,35.7,80.5,48.9,16.0,0.0,1.9,0.339747
1598,peanuts cooked,572,24.3,38.3,39.6,15.8,4.4,1.4,0.323199
1102,navy beans canned,296,19.7,53.6,1.1,13.4,0.7,1.2,0.320707
122,refried red beans,336,11.7,36.0,16.1,11.0,0.0,0.9,0.294112
1939,taco flavor tortilla chips,1090,17.9,143.2,54.9,12.0,0.0,1.8,0.248747
143,baked beans,310,11.1,43.3,10.3,11.0,0.0,0.8,0.240084
1098,lima beans canned,190,11.9,35.9,0.4,11.6,0.0,0.8,0.236779


Predict Nutrition

In [59]:
model = joblib.load("../models/nutrition_model.pkl")
def predict_nutrition(patient_df):

    prediction = model.predict(patient_df)

    return {
        "caloric_value": prediction[0][0],
        "protein": prediction[0][1],
        "carbohydrates": prediction[0][2],
        "fat": prediction[0][3],
        "dietary_fiber": prediction[0][4],
        "sugars": prediction[0][5],
        "sodium": prediction[0][6]
    }

Candidate Generation

split it into three parts

1. Health Filter

2. Diet Filter

3. Allergy Filter

In [60]:
def apply_health_rules(food_df, patient, nutrition):

    condition = patient["health_condition"].iloc[0]

    filtered = food_df.copy()

    if condition == "Diabetes":

        filtered = filtered[
            (filtered["sugars"] <= nutrition["sugars"] * 1.2) &
            (filtered["dietary_fiber"] >= nutrition["dietary_fiber"] * 0.3)
        ]

    elif condition == "Hypertension":

        filtered = filtered[
            filtered["sodium"] <= nutrition["sodium"] * 1.1
        ]

    elif condition == "Obesity":

        filtered = filtered[
            filtered["caloric_value"] <= nutrition["caloric_value"] * 0.6
        ]

    elif condition == "Heart Disease":

        filtered = filtered[
            filtered["cholesterol"] <= 100
        ]

    return filtered

In [61]:
NON_VEG = [
    "chicken",
    "beef",
    "fish",
    "pork",
    "lamb",
    "turkey",
    "duck",
    "shrimp",
    "crab",
    "mutton",
    "bacon"
]

In [62]:
def apply_diet_filter(food_df, patient):

    diet = patient["diet_type"].iloc[0]

    if diet == "Vegetarian":

        pattern = "|".join(NON_VEG)

        food_df = food_df[
            ~food_df["food"].str.lower().str.contains(
                pattern,
                na=False
            )
        ]

    return food_df

In [63]:
ALLERGY_MAP = {

    "Nut Allergy":[
        "almond",
        "cashew",
        "walnut",
        "peanut"
    ],

    "Seafood Allergy":[
        "fish",
        "shrimp",
        "crab",
        "salmon",
        "tuna"
    ],

    "Dairy Allergy":[
        "milk",
        "cheese",
        "cream",
        "butter"
    ]
}

In [64]:
def apply_allergy_filter(food_df, patient):

    allergy = patient["allergy"].iloc[0]

    if allergy == "None":

        return food_df

    if allergy not in ALLERGY_MAP:

        return food_df

    pattern = "|".join(ALLERGY_MAP[allergy])

    return food_df[
        ~food_df["food"].str.lower().str.contains(
            pattern,
            na=False
        )
    ]

Combine all filters

In [76]:
def generate_candidates(food_df, patient, nutrition):

    foods = apply_health_rules(
        food_df,
        patient,
        nutrition
    )

    foods = apply_diet_filter(
        foods,
        patient
    )

    foods = apply_allergy_filter(
        foods,
        patient
    )

    foods = apply_quality_filter(
    foods
)

    return foods

Ranking

In [68]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

def rank_foods(candidate_foods, nutrition, scaler):

    nutrition_features = [
        "caloric_value",
        "protein",
        "carbohydrates",
        "fat",
        "dietary_fiber",
        "sugars",
        "sodium"
    ]

    # Patient vector
    patient_vector = scaler.transform(
        pd.DataFrame([nutrition])
    )

    # Food vectors
    food_vectors = scaler.transform(
        candidate_foods[nutrition_features]
    )

    # Cosine similarity
    similarities = cosine_similarity(
        patient_vector,
        food_vectors
    )[0]

    ranked = candidate_foods.copy()

    ranked["similarity_score"] = similarities

    # Nutrition distance
    ranked["nutrition_score"] = (
        abs(ranked["caloric_value"] - nutrition["caloric_value"]) +
        abs(ranked["protein"] - nutrition["protein"]) +
        abs(ranked["carbohydrates"] - nutrition["carbohydrates"]) +
        abs(ranked["fat"] - nutrition["fat"]) +
        abs(ranked["dietary_fiber"] - nutrition["dietary_fiber"])
    )

    # Normalize scores
    score_scaler = MinMaxScaler()

    ranked["similarity_norm"] = score_scaler.fit_transform(
        ranked[["similarity_score"]]
    )

    ranked["nutrition_norm"] = 1 - score_scaler.fit_transform(
        ranked[["nutrition_score"]]
    )

    # Final score
    ranked["final_score"] = (
        0.7 * ranked["similarity_norm"] +
        0.3 * ranked["nutrition_norm"]
    )

    ranked = ranked.sort_values(
        "final_score",
        ascending=False
    )

    return ranked

The ranking function knows absolutely nothing about

diabetes

allergies

diet

Its only job is ranking.

Final Pipeline

In [79]:
nutrition = predict_nutrition(sample_patient)

candidate_foods = generate_candidates(
    recommendation_df,
    sample_patient,
    nutrition
)

recommendations = rank_foods(
    candidate_foods,
    nutrition,
    scaler
)

recommendations.head(10)

,food,caloric_value,protein,carbohydrates,fat,dietary_fiber,sugars,sodium,similarity_score,calorie_diff,protein_diff,carb_diff,fat_diff,fiber_diff,nutrition_score,similarity_normalized,nutrition_normalized,final_score,similarity_norm,nutrition_norm
193,black bean soup,234,12.4,39.6,3.4,17.5,6.4,2.500,0.563319,1466.0,77.6,130.4,51.6,17.5,1743.1,0.707595,0.149784,0.732955,1.000000,0.109849
188,chili with beans canned,287,14.6,30.5,14.1,11.3,3.0,1.300,0.463632,1413.0,75.4,139.5,40.9,23.7,1692.5,0.640613,0.178547,0.643730,0.856220,0.147920
1598,peanuts cooked,572,24.3,38.3,39.6,15.8,4.4,1.400,0.323199,1128.0,65.7,131.7,15.4,19.2,1360.0,0.546253,0.367553,0.576997,0.653673,0.398089
1102,navy beans canned,296,19.7,53.6,1.1,13.4,0.7,1.200,0.320707,1404.0,70.3,116.4,53.9,21.6,1666.2,0.544579,0.193497,0.505367,0.650079,0.167707
122,refried red beans,336,11.7,36.0,16.1,11.0,0.0,0.900,0.294112,1364.0,78.3,134.0,38.9,24.0,1639.2,0.526709,0.208845,0.484610,0.611720,0.188022
1633,sunflower seeds toasted,829,23.1,27.6,76.1,15.4,0.0,0.800,0.125185,871.0,66.9,142.4,21.1,19.6,1121.0,0.413203,0.503411,0.431025,0.368075,0.577910
143,baked beans,310,11.1,43.3,10.3,11.0,0.0,0.800,0.240084,1390.0,78.9,126.7,44.7,24.0,1664.3,0.490407,0.194577,0.424398,0.533795,0.169137
1009,coconut meat,1405,13.2,60.5,133.0,35.7,24.7,0.022,-0.024244,295.0,76.8,109.5,78.0,0.7,560.0,0.312799,0.822306,0.406786,0.152552,1.000000
1650,spanish peanuts roasted,851,41.2,25.7,72.1,13.1,0.0,0.600,0.082731,849.0,48.8,144.3,17.1,21.9,1081.1,0.384677,0.526091,0.397169,0.306842,0.607930
1098,lima beans canned,190,11.9,35.9,0.4,11.6,0.0,0.800,0.236779,1510.0,78.1,134.1,54.6,23.4,1800.2,0.488186,0.117326,0.390386,0.529028,0.066887


In [78]:
def recommend_foods(patient_df, model, food_df, scaler, top_n=5):

    nutrition = predict_nutrition(model, patient_df)

    candidates = generate_candidates(
        food_df,
        patient_df,
        nutrition
    )

    ranked = rank_foods(
        candidates,
        nutrition,
        scaler
    )
    ranked = ranked.drop(
    columns=[
        "calorie_diff",
        "protein_diff",
        "carb_diff",
        "fat_diff",
        "fiber_diff",
        "similarity_norm",
        "nutrition_norm"
    ],
    errors="ignore"
)

    return ranked[
    [
        "food",
        "caloric_value",
        "protein",
        "carbohydrates",
        "fat",
        "dietary_fiber",
        "similarity_score",
        "final_score"
    ]
].head(top_n)

In [74]:
LOW_QUALITY_FOODS = [
    "chips",
    "fries",
    "burger",
    "pizza",
    "taco bell",
    "cola",
    "soft drink",
    "candy",
    "cookie",
    "cookies",
    "cake",
    "chocolate",
    "doughnut",
    "donut",
    "ice cream"
]

In [75]:
def apply_quality_filter(food_df):

    pattern = "|".join(LOW_QUALITY_FOODS)

    return food_df[
        ~food_df["food"].str.lower().str.contains(
            pattern,
            na=False
        )
    ]